# CultureConnect — Data Analysis

Exploratory data analysis of the CultureConnect platform — a community marketplace connecting residents, SME businesses, and local council across Hertfordshire.

**Questions explored:**
- Which areas are most culturally active?
- Which categories attract the most bookings and revenue?
- How are users distributed across roles and areas?
- Which listings have the highest customer satisfaction?
- What does the revenue trend look like over time?

In [ ]:
!pip install -q pandas matplotlib seaborn numpy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import datetime, timedelta
import random

random.seed(42)
np.random.seed(42)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13

print('Libraries loaded')

## 1. Generate Sample Data
The schema defines the structure. Here we generate realistic sample data to analyse.

In [ ]:
# --- Reference tables ---
areas = pd.DataFrame({
    'area_id': range(1, 7),
    'area_name': ['Hertfordshire North', 'Hertfordshire South', 'Hertfordshire East',
                  'Hertfordshire West', 'Hertfordshire Town Centre', 'Hertfordshire Central']
})

categories = pd.DataFrame({
    'category_id': range(1, 9),
    'category_name': ['Visual Arts', 'Music', 'Performing Arts', 'Creative Media',
                      'Literature', 'Cultural Merchandise', 'Sports & Recreation', 'Wellbeing & Community']
})

roles = pd.DataFrame({
    'role_id': [1, 2, 3, 4],
    'role_name': ['Resident', 'SME', 'Council Member', 'Council Administrator']
})

# --- Users ---
n_users = 120
role_weights = [0.65, 0.25, 0.08, 0.02]
users = pd.DataFrame({
    'user_ref_no': range(1, n_users + 1),
    'role_id': np.random.choice([1, 2, 3, 4], size=n_users, p=role_weights),
    'area_id': np.random.choice(range(1, 7), size=n_users),
    'status': np.random.choice(['active', 'inactive', 'approved'], size=n_users, p=[0.70, 0.10, 0.20]),
    'created_at': [datetime(2025, 9, 1) + timedelta(days=random.randint(0, 270)) for _ in range(n_users)]
})

# --- SME Businesses ---
sme_users = users[users['role_id'] == 2]['user_ref_no'].tolist()
sme_businesses = pd.DataFrame({
    'sme_id': range(1, len(sme_users) + 1),
    'user_ref_no': sme_users,
    'category_id': np.random.choice(range(1, 9), size=len(sme_users)),
    'approval_status': np.random.choice(['approved', 'pending', 'rejected'], size=len(sme_users), p=[0.70, 0.20, 0.10]),
    'area_id': np.random.choice(range(1, 7), size=len(sme_users))
})

# --- Listings ---
approved_smes = sme_businesses[sme_businesses['approval_status'] == 'approved']['sme_id'].tolist()
n_listings = 90
listings = pd.DataFrame({
    'listing_id': range(1, n_listings + 1),
    'sme_id': np.random.choice(approved_smes, size=n_listings),
    'category_id': np.random.choice(range(1, 9), size=n_listings),
    'price': np.round(np.random.uniform(5, 150, size=n_listings), 2),
    'status': np.random.choice(['active', 'inactive'], size=n_listings, p=[0.80, 0.20]),
    'created_at': [datetime(2025, 10, 1) + timedelta(days=random.randint(0, 240)) for _ in range(n_listings)]
})

# --- Service Bookings ---
resident_users = users[users['role_id'] == 1]['user_ref_no'].tolist()
active_listings = listings[listings['status'] == 'active']['listing_id'].tolist()
n_bookings = 200
bookings = pd.DataFrame({
    'booking_id': range(1, n_bookings + 1),
    'user_ref_no': np.random.choice(resident_users, size=n_bookings),
    'listing_id': np.random.choice(active_listings, size=n_bookings),
    'booking_date': [datetime(2025, 11, 1) + timedelta(days=random.randint(0, 210)) for _ in range(n_bookings)],
    'status': np.random.choice(['confirmed', 'completed', 'cancelled'], size=n_bookings, p=[0.20, 0.65, 0.15])
})

# --- Orders ---
n_orders = 150
orders = pd.DataFrame({
    'order_id': range(1, n_orders + 1),
    'user_ref_no': np.random.choice(resident_users, size=n_orders),
    'total_amount': np.round(np.random.uniform(10, 300, size=n_orders), 2),
    'order_date': [datetime(2025, 11, 1) + timedelta(days=random.randint(0, 210)) for _ in range(n_orders)],
    'status': np.random.choice(['completed', 'pending', 'cancelled'], size=n_orders, p=[0.70, 0.20, 0.10])
})

# --- Reviews ---
n_reviews = 160
reviews = pd.DataFrame({
    'review_id': range(1, n_reviews + 1),
    'user_ref_no': np.random.choice(resident_users, size=n_reviews),
    'listing_id': np.random.choice(active_listings, size=n_reviews),
    'rating': np.random.choice([1, 2, 3, 4, 5], size=n_reviews, p=[0.05, 0.08, 0.17, 0.35, 0.35]),
    'created_at': [datetime(2025, 11, 1) + timedelta(days=random.randint(0, 210)) for _ in range(n_reviews)]
})

print(f'Users: {len(users)} | SMEs: {len(sme_businesses)} | Listings: {len(listings)}')
print(f'Bookings: {len(bookings)} | Orders: {len(orders)} | Reviews: {len(reviews)}')

## 2. User Distribution

In [ ]:
user_roles = users.merge(roles, on='role_id')
role_counts = user_roles['role_name'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar chart
role_counts.plot(kind='bar', ax=axes[0], color=sns.color_palette('muted'), edgecolor='white')
axes[0].set_title('Number of Users by Role')
axes[0].set_xlabel('')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

# Pie chart
axes[1].pie(role_counts, labels=role_counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('muted'), startangle=140)
axes[1].set_title('User Role Breakdown')

plt.tight_layout()
plt.show()
print(role_counts.to_string())

In [ ]:
user_areas = users.merge(areas, on='area_id')
area_counts = user_areas['area_name'].value_counts()

ax = area_counts.plot(kind='barh', color=sns.color_palette('muted'), edgecolor='white')
ax.set_title('Number of Users per Area')
ax.set_xlabel('Number of Users')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## 3. Listings Analysis

In [ ]:
listing_cats = listings.merge(categories, on='category_id')
cat_counts = listing_cats['category_name'].value_counts()

ax = cat_counts.plot(kind='bar', color=sns.color_palette('pastel'), edgecolor='white')
ax.set_title('Number of Listings per Category')
ax.set_xlabel('')
ax.set_ylabel('Listings')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()

In [ ]:
avg_price = listing_cats.groupby('category_name')['price'].mean().sort_values(ascending=False)

ax = avg_price.plot(kind='bar', color=sns.color_palette('coolwarm', len(avg_price)), edgecolor='white')
ax.set_title('Average Listing Price by Category')
ax.set_xlabel('')
ax.set_ylabel('Average Price (£)')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()

## 4. Bookings Analysis

In [ ]:
booking_listings = bookings.merge(listings[['listing_id', 'category_id']], on='listing_id')
booking_cats = booking_listings.merge(categories, on='category_id')
booking_by_cat = booking_cats['category_name'].value_counts()

ax = booking_by_cat.plot(kind='bar', color=sns.color_palette('Set2'), edgecolor='white')
ax.set_title('Total Bookings per Category')
ax.set_xlabel('')
ax.set_ylabel('Bookings')
ax.tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.show()

In [ ]:
bookings['month'] = bookings['booking_date'].dt.to_period('M')
monthly_bookings = bookings.groupby('month').size()

ax = monthly_bookings.plot(kind='line', marker='o', color='steelblue', linewidth=2)
ax.set_title('Monthly Booking Trend')
ax.set_xlabel('Month')
ax.set_ylabel('Number of Bookings')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Revenue Analysis

In [ ]:
completed_orders = orders[orders['status'] == 'completed'].copy()
completed_orders['month'] = completed_orders['order_date'].dt.to_period('M')
monthly_revenue = completed_orders.groupby('month')['total_amount'].sum()

ax = monthly_revenue.plot(kind='bar', color='mediumseagreen', edgecolor='white')
ax.set_title('Monthly Revenue (Completed Orders)')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (£)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'£{x:,.0f}'))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f'Total revenue: £{completed_orders["total_amount"].sum():,.2f}')
print(f'Average order value: £{completed_orders["total_amount"].mean():,.2f}')

## 6. Customer Satisfaction

In [ ]:
rating_counts = reviews['rating'].value_counts().sort_index()

colors = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#27ae60']
ax = rating_counts.plot(kind='bar', color=colors, edgecolor='white')
ax.set_title('Review Rating Distribution')
ax.set_xlabel('Rating (1 = Poor, 5 = Excellent)')
ax.set_ylabel('Number of Reviews')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

print(f'Average rating: {reviews["rating"].mean():.2f} / 5')

In [ ]:
review_cats = reviews.merge(listings[['listing_id', 'category_id']], on='listing_id')
review_cats = review_cats.merge(categories, on='category_id')
avg_rating_cat = review_cats.groupby('category_name')['rating'].mean().sort_values(ascending=False)

ax = avg_rating_cat.plot(kind='barh', color=sns.color_palette('RdYlGn', len(avg_rating_cat)), edgecolor='white')
ax.set_title('Average Customer Rating by Category')
ax.set_xlabel('Average Rating')
ax.set_xlim(0, 5)
ax.axvline(x=4, color='gray', linestyle='--', alpha=0.5, label='4.0 threshold')
ax.legend()
plt.tight_layout()
plt.show()

## 7. SME Business Overview

In [ ]:
approval_counts = sme_businesses['approval_status'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

approval_counts.plot(kind='bar', ax=axes[0], color=['#2ecc71', '#f39c12', '#e74c3c'], edgecolor='white')
axes[0].set_title('SME Application Status')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)

sme_area = sme_businesses.merge(areas, on='area_id')['area_name'].value_counts()
sme_area.plot(kind='barh', ax=axes[1], color=sns.color_palette('muted'), edgecolor='white')
axes[1].set_title('SME Businesses per Area')
axes[1].set_xlabel('Count')

plt.tight_layout()
plt.show()

approval_rate = (approval_counts.get('approved', 0) / len(sme_businesses)) * 100
print(f'SME approval rate: {approval_rate:.1f}%')

## 8. Summary Insights


In [ ]:
print('===== CULTURECONNECT PLATFORM SUMMARY =====')
print(f'Total users:           {len(users)}')
print(f'Active residents:      {len(users[users["role_id"] == 1])}')
print(f'Registered SMEs:       {len(sme_businesses)}')
print(f'Approved SMEs:         {len(sme_businesses[sme_businesses["approval_status"] == "approved"])}')
print(f'Active listings:       {len(listings[listings["status"] == "active"])}')
print(f'Total bookings:        {len(bookings)}')
print(f'Completed orders:      {len(completed_orders)}')
print(f'Total revenue:         £{completed_orders["total_amount"].sum():,.2f}')
print(f'Total reviews:         {len(reviews)}')
print(f'Average rating:        {reviews["rating"].mean():.2f} / 5')
print()
print('Top category by bookings:', booking_by_cat.index[0])
print('Top category by avg price:', avg_price.index[0])
print('Most active area (users):', area_counts.index[0])